In [7]:
import os
import glob
import json

# Get sorted file paths and file names
file_paths1 = glob.glob('/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/*')
file_paths1.sort()

file_names1 = [os.path.basename(path) for path in file_paths1]
file_names1.sort()

# Initialize list to store valid data entries
file_list = []

# Iterate over each folder and check if necessary files exist
for folder_path, folder_name in zip(file_paths1, file_names1):
    # Construct full paths for all modalities and label
    t1c_path = os.path.join(folder_path, folder_name + '-t1c.nii.gz')
    t1n_path = os.path.join(folder_path, folder_name + '-t1n.nii.gz')
    t2f_path = os.path.join(folder_path, folder_name + '-t2f.nii.gz')
    t2w_path = os.path.join(folder_path, folder_name + '-t2w.nii.gz')
    label_path = os.path.join(folder_path, folder_name + '-seg.nii.gz')

    # Check if all required files exist
    if all(os.path.isfile(p) for p in [t1c_path, t1n_path, t2f_path, t2w_path, label_path]):
        file_list.append({
            "image": [t1c_path, t1n_path, t2f_path, t2w_path],
            "label": label_path
        })
    else:
        print(f"Skipping empty or incomplete directory: {folder_path}")

# Save the list to a JSON file
file_json = {
    "training": file_list
}

file_path = '/home/ali/R_and_D/BTS/2024/dataset/preprocessed_2024_dataset.json'
with open(file_path, 'w') as json_file:
    json.dump(file_json, json_file, indent=4)

# Print first 10 entries
print(json.dumps({"training": file_list[:10]}, indent=4))


{
    "training": [
        {
            "image": [
                "/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t1c.nii.gz",
                "/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t1n.nii.gz",
                "/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t2f.nii.gz",
                "/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t2w.nii.gz"
            ],
            "label": "/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-seg.nii.gz"
        },
        {
            "image": [
                "/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/BraTS-GLI-00005-101/BraTS-GLI-00005-101-t1c.nii.gz",
                "/home/ali/R_and_D/BTS/2024/dataset/training/training_data1_v2/BraTS-GLI-0000

In [ ]:
import pytorch_lightning
import monai
import numpy as np
import pandas as pd
from typing import Union, List
from PIL import Image
from monai.utils import set_determinism
from monai.utils.enums import MetricReduction
from monai.transforms import (
    AsDiscrete,
    ScaleIntensityRangePercentilesd,
    EnsureChannelFirstd,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    ScaleIntensityRanged,
    Spacingd,
    EnsureType,
    MapTransform,
    Activations,
    Activationsd,
    Invertd,
    NormalizeIntensityd,
    RandFlipd,
    RandScaleIntensityd,
    RandShiftIntensityd,
    RandSpatialCropd,
    EnsureTyped,
    SpatialPadd,
    SaveImage,
    RandRotate90d,
    ConcatItemsd,
    DeleteItemsd,
    RandAffined,
    RandGaussianNoised,
    RandAdjustContrastd,
)
from monai.networks.utils import one_hot
from monai.networks.nets import SwinUNETR, UNETR, SegResNet
from monai.networks.layers import Norm
from monai.metrics import DiceMetric, HausdorffDistanceMetric, ConfusionMatrixMetric, compute_hausdorff_distance, CumulativeIterationMetric
from monai.losses import DiceCELoss, DiceLoss, TverskyLoss
from monai.inferers import sliding_window_inference
from monai.data import PersistentDataset, list_data_collate, decollate_batch, DataLoader, load_decathlon_datalist, CacheDataset
from monai.config import print_config
from monai.apps import download_and_extract, DecathlonDataset
from monai.handlers.utils import from_engine
import torch
import matplotlib.pyplot as plt
import tempfile
import shutil
import os
import glob
import nibabel as nib
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks.timer import Timer
from sklearn.model_selection import train_test_split
from monai.transforms import MapTransform
import torch.nn as nn
import torch.nn.functional as F

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
print_config()

class ConvertToOverlapLabels(MapTransform):
    def __init__(self, keys):
        super().__init__(keys)
        self.keys = keys

    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            label = d[key]
            # Create masks
            et_mask = (label == 3)
            tc_mask = (label == 1) | et_mask
            wt_mask = tc_mask | (label == 2)

            # Initialize new label map
            new_label = torch.zeros_like(label, dtype=torch.int64)  # Ensure integer labels

            # Assign values in order of priority
            new_label[wt_mask] = 1  # WT (lowest priority)
            new_label[tc_mask] = 2  # TC (higher priority)
            new_label[et_mask] = 3  # ET (highest priority)

            d[key] = new_label

        return d

# ------------------------
# Mixture of Experts with SWINUNETR
# ------------------------

class Expert(nn.Module):
    """Individual expert network using SWINUNETR architecture"""
    def __init__(self, img_size=(96, 96, 96), in_channels=4, out_channels=4):
        super().__init__()
        self.model = SwinUNETR(
            img_size=img_size,
            in_channels=in_channels,
            out_channels=out_channels,
            feature_size=48,
            use_checkpoint=True,
        )
        
    def forward(self, x):
        return self.model(x)

class GatingNetwork(nn.Module):
    """Gating network that learns to weight the experts"""
    def __init__(self, num_experts=3, in_channels=4, hidden_dim=64):
        super().__init__()
        self.num_experts = num_experts
        
        # Shared feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Conv3d(in_channels, hidden_dim, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten()
        )
        
        # Gating layers
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_experts),
            nn.Softmax(dim=1)
        )
        
    def forward(self, x):
        features = self.feature_extractor(x)
        return self.gate(features)

class MoE_SWINUNETR(nn.Module):
    """Mixture of Experts with SWINUNETR architecture"""
    def __init__(self, num_experts=3, img_size=(96, 96, 96), in_channels=4, out_channels=4):
        super().__init__()
        self.num_experts = num_experts
        self.img_size = img_size
        
        # Create multiple experts
        self.experts = nn.ModuleList([Expert(img_size, in_channels, out_channels) for _ in range(num_experts)])
        
        # Gating network
        self.gate = GatingNetwork(num_experts, in_channels)
        
        # Final convolution to combine expert outputs
        self.final_conv = nn.Conv3d(out_channels * num_experts, out_channels, kernel_size=1)
        
    def forward(self, x):
        # Get gating weights
        gate_weights = self.gate(x)  # [B, num_experts]
        
        # Get expert outputs
        expert_outputs = []
        for i, expert in enumerate(self.experts):
            expert_out = expert(x)  # [B, C, D, H, W]
            expert_outputs.append(expert_out * gate_weights[:, i].view(-1, 1, 1, 1, 1))
        
        # Combine expert outputs
        combined = torch.cat(expert_outputs, dim=1)  # [B, C*num_experts, D, H, W]
        output = self.final_conv(combined)
        
        return output

# ------------------------
# Lightning Module
# ------------------------

class Net(pytorch_lightning.LightningModule):
    def __init__(self):
        super().__init__()
        
        # Initialize the MoE model
        self._model = MoE_SWINUNETR(
            num_experts=3,  # Number of experts
            img_size=(96, 96, 96),
            in_channels=4,
            out_channels=4
        ).to(device)

        self.loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
        self.post_pred = AsDiscrete(argmax=True, to_onehot=4)
        self.post_label = AsDiscrete(to_onehot=4)

        self.dice_metric = DiceMetric(include_background=False, reduction="mean")
        self.dice_metric_batch = DiceMetric(include_background=False, reduction="mean_batch")
        self.haursdoff = HausdorffDistanceMetric(include_background=False, distance_metric='euclidean',
                                               percentile=None, directed=False, reduction="mean_batch", get_not_nans=True)
        self.check_val = 1

        self.best_val_dice = 0
        self.best_val_epoch = 0
        self.epoch_loss_values = []
        self.metric_values = []
        self.metric_values_ncr = []
        self.metric_values_ed = []
        self.metric_values_et = []
        self.metric_values_wt = []
        self.metric_values_tc = []
        self.metric_values_back = []

        self.haursdoff_values_ncr = []
        self.haursdoff_values_ed = []
        self.haursdoff_values_et = []
        self.haursdoff_values_back = []

        self.validation_step_outputs = []
        self.training_step_outputs = []

    def forward(self, x):
        return self._model(x)

    def prepare_data(self):
        """Loads dataset and splits all files into training and validation sets."""
        # Load dataset
        datasets = "/home/ali/R_and_D/BTS/2024/dataset/preprocessed_2024_dataset.json"
        datalist = load_decathlon_datalist(datasets, True, "training")
        print(f"Total samples in dataset: {len(datalist)}")

        # Split dataset into training (80%) and validation (20%) randomly
        train_files, val_files = train_test_split(datalist, test_size=0.1, random_state=42)
        # train_files = train_files[:int(1 * len(train_files))]
        # val_files = val_files[:int(1 * len(val_files))]

        # Setting deterministic training for reproducibility
        set_determinism(seed=0)

        # Defining the data transforms
        train_transform = Compose([
            LoadImaged(keys=["image", "label"]),
            EnsureChannelFirstd(keys=["image", "label"]),
            Orientationd(keys=["image", "label"], axcodes="RAS"),
            ConvertToOverlapLabels(keys=["label"]),
            Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
            NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
            RandSpatialCropd(keys=["image", "label"], roi_size=[96, 96, 96], random_size=False),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
            RandScaleIntensityd(keys="image", factors=0.1, prob=1.0),
            RandShiftIntensityd(keys="image", offsets=0.1, prob=1.0),
        ])

        val_transform = Compose([
            LoadImaged(keys=["image", "label"]),
            EnsureChannelFirstd(keys=["image", "label"]),
            Orientationd(keys=["image", "label"], axcodes="RAS"),
            ConvertToOverlapLabels(keys=["label"]),
            Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
            NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
        ])

        self.train_ds = monai.data.Dataset(
            data=train_files,
            transform=train_transform,
        )

        self.val_ds = monai.data.Dataset(
            data=val_files,
            transform=val_transform,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=2,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=2,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
        )

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self._model.parameters(), lr=1e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6)

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            },
        }

    def training_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        raw_outputs = self.forward(images)

        # Compute Loss
        loss = self.loss_function(raw_outputs, labels)

        # Post-process predictions
        outputs = [self.post_pred(i) for i in decollate_batch(raw_outputs)]
        labels = [self.post_label(i) for i in decollate_batch(labels)]

        # Compute Metrics
        self.dice_metric(y_pred=outputs, y=labels)
        self.dice_metric_batch(y_pred=outputs, y=labels)

        # Log Metrics
        self.log("train_loss", loss.item(), prog_bar=True, logger=True)

        # Store loss values for epoch-end calculations
        self.training_step_outputs.append({"loss": loss, "dice": self.dice_metric.aggregate().mean()})
        return {"loss": loss}

    def on_train_epoch_end(self):
        # Compute average training loss over the epoch
        avg_loss = torch.stack([x["loss"] for x in self.training_step_outputs]).mean()
        avg_dice = torch.stack([x["dice"] for x in self.training_step_outputs]).mean()

        self.epoch_loss_values.append(avg_loss.detach().cpu().numpy())

        # Print Training Loss and Training Dice at the end of the epoch
        print(f"Epoch {self.current_epoch}: Training Loss: {avg_loss.item():.4f}, Training Dice: {avg_dice.item():.4f}")

        # Clear the outputs list
        self.training_step_outputs.clear()

    def validation_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        roi_size, sw_batch_size = (96, 96, 96), 4
        outputs = sliding_window_inference(images, roi_size, sw_batch_size, self.forward)

        loss = self.loss_function(outputs, labels)
        outputs = [self.post_pred(i) for i in decollate_batch(outputs)]
        labels = [self.post_label(i) for i in decollate_batch(labels)]

        # Compute Metrics
        self.dice_metric(y_pred=outputs, y=labels)
        self.dice_metric_batch(y_pred=outputs, y=labels)

        self.validation_step_outputs.append({"val_loss": loss, "val_number": len(outputs)})
        return {"val_loss": loss}

    def on_validation_epoch_end(self):
        val_loss, num_items = 0, 0
        for output in self.validation_step_outputs:
            val_loss += output["val_loss"].sum().item()
            num_items += output["val_number"]

        mean_val_dice = self.dice_metric.aggregate().item()
        self.metric_values.append(np.array(mean_val_dice))
        self.dice_metric.reset()

        metric_batch = self.dice_metric_batch.aggregate()
        metric_wt, metric_tc, metric_et = metric_batch[0].item(), metric_batch[1].item(), metric_batch[2].item()
        self.metric_values_ncr.append(metric_wt)
        self.metric_values_ed.append(metric_tc)
        self.metric_values_et.append(metric_et)
        self.dice_metric_batch.reset()

        mean_val_loss = torch.tensor(val_loss / num_items)

        # Print validation loss and metrics in a single line
        print(
            f"Epoch {self.current_epoch}: Val Loss: {mean_val_loss.item():.4f}, "
            f"Mean Dice: {mean_val_dice:.4f} "
            f"(at epoch {self.best_val_epoch}), NCR: {metric_wt:.4f}, "
            f"ED: {metric_tc:.4f}, ET: {metric_et:.4f}"
        )

        if mean_val_dice > self.best_val_dice:
            self.best_val_dice = mean_val_dice
            self.best_val_epoch = self.current_epoch
            torch.save(self._model, f"Model_MoE_SWINUNETR24.pt")

        self.log("val_loss", mean_val_loss.item())
        self.validation_step_outputs.clear()
        return {"val_loss": mean_val_loss}

# Training and Validation Setup
root_dir = "/home/ali/R_and_D/BTS/2024/model/"
net = Net()

# Set up checkpoints
checkpoint_callback = ModelCheckpoint(dirpath=root_dir, filename="best_2024_model", save_last=True)
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=0.00,
    patience=10,
    verbose=False,
    mode='min'
)

# Initialize Lightning's trainer
trainer = pytorch_lightning.Trainer(
    precision="16-mixed",
    accelerator='gpu',
    devices="auto",
    max_epochs=130,
    check_val_every_n_epoch=net.check_val,
    callbacks=[checkpoint_callback, early_stop_callback],
    default_root_dir=root_dir,
    limit_val_batches=20,
    num_sanity_val_steps=0,
    gradient_clip_val=1.0,
)

# Training
trainer.fit(net)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type          | Params | Mode 
--------------------------------------------------------
0 | _model        | MoE_SWINUNETR | 186 M  | train
1 | loss_function | DiceCELoss    | 0      | train
--------------------------------------------------------
186 M     Trainable params
0         Non-trainable params
186 M     Total params
746.349   Total estimated model params size (MB)
831       Modules in train mode
0         Modules in eval mode


Total samples in dataset: 1350
Epoch 114: 100%|██████████| 608/608 [06:19<00:00,  1.60it/s, v_num=0, train_loss=0.313]


In [47]:
# Plot the best validation scores
print(f"\nBest Validation Mean Dice: {net.best_val_dice:.4f} ")
print(f"Best WT Dice: {max(net.metric_values_ncr):.4f}")
print(f"Best TC Dice: {max(net.metric_values_ed):.4f}")
print(f"Best ET Dice: {max(net.metric_values_et):.4f}")


Best Validation Mean Dice: 0.8456 
Best WT Dice: 0.8894
Best TC Dice: 0.7367
Best ET Dice: 0.8146
